# Pyrates FEP
# Notebook 01: data cleaning

First step of the Pyrates_FEP learning analytics pipeline. It starts from the raw export of
interaction traces (`data/imported/imported_interaction_traces.xlsx`) and produces a clean
dataset (`data/cleaned/cleaned_interaction_traces.csv`) in the exact CSV format expected by the
webtool importer.

Three fixes are applied, **all of them limited to the `code` column**; no row is filtered out
and no column is dropped except the spreadsheet's own index artifact.

**Fix 0: Excel formula cells.**
When the text copied by a student starts with `=`, Excel stored the cell as a *formula* rather
than as text. Reading the workbook with cached values returns nothing for those cells (the
computed value is not stored in the file), so the original text would be silently lost. The
literal formula text is still there and is read back with `data_only=False`.

**Fix 1: mojibake repair.**
A batch of LLM feedback messages was exported with CP1252 punctuation that got misinterpreted as
raw Unicode C1 control characters instead of being decoded through Windows-1252, corrupting the
surrounding text. The mapping below was verified against the webtool export (0 mismatches):

| Corrupted character | Replacement | Origin |
|---|---|---|
| `U+0092` | `'` (straight apostrophe) | mis-decoded curly apostrophe |
| `U+0085` | `...` (three literal periods) | mis-decoded ellipsis |
| `U+009C` | `œ` | mis-decoded oe ligature |

**Fix 2: copy/paste backfill.**
For a number of `COPIED` and `PASTED > CONTENT` rows the `code` field was captured empty even
though the student clearly copied or pasted real text: a capture-timing issue in the game client
(the clipboard/selection was not readable yet when the event fired), not a webtool bug. Those rows
are backfilled from the matching trace of the same game (an empty copy from the next paste, an
empty paste from the previous copy), and only when the attribution is unambiguous. See section 4
for the exact rule.

## 1. Setup and imports

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import openpyxl
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Paths relative to the notebook: notebooks/ -> project root -> data/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
INPUT_PATH = DATA_DIR / "imported" / "imported_interaction_traces.xlsx"
OUTPUT_PATH = DATA_DIR / "cleaned" / "cleaned_interaction_traces.csv"

# Paths are printed relative to the project root, so that no local path ends up in the outputs.
print("Project root :", PROJECT_ROOT.name)
print("Input        :", INPUT_PATH.relative_to(PROJECT_ROOT), "| exists:", INPUT_PATH.exists())
print("Output       :", OUTPUT_PATH.relative_to(PROJECT_ROOT))

## 2. Loading the raw data

### 2.1 Cell values

In [ ]:
df_raw = pd.read_excel(INPUT_PATH)

# The spreadsheet carries a leading unnamed index column, which is not part of the schema.
INDEX_ARTIFACT = "Unnamed: 0"
if INDEX_ARTIFACT in df_raw.columns:
    df_raw = df_raw.drop(columns=[INDEX_ARTIFACT])

print(f"{len(df_raw):,} rows x {df_raw.shape[1]} columns")
df_raw.head(3)

### 2.2 Fix 0: recovering the `code` cells stored as Excel formulas

In [ ]:
def read_formula_code_cells(path: Path) -> dict[int, str]:
    # Return {trace id: literal formula text} for every `code` cell Excel stored as a formula.
    workbook = openpyxl.load_workbook(path, read_only=True, data_only=False)
    sheet = workbook.active
    rows = sheet.iter_rows(values_only=False)
    header = [cell.value for cell in next(rows)]
    id_position, code_position = header.index("id"), header.index("code")

    formulas: dict[int, str] = {}
    for row in rows:
        cell = row[code_position]
        if cell.data_type == "f" and isinstance(cell.value, str):
            formulas[row[id_position].value] = cell.value

    workbook.close()
    return formulas


formula_cells = read_formula_code_cells(INPUT_PATH)
print(f"`code` cells stored as formulas: {len(formula_cells)}")

In [ ]:
df = df_raw.copy()
row_of_id = pd.Series(df.index, index=df["id"])

recovered_cells = []
for trace_id, formula_text in formula_cells.items():
    row_index = row_of_id[trace_id]
    recovered_cells.append(
        {
            "id": trace_id,
            "action": f"{df.at[row_index, 'action_name']} > {df.at[row_index, 'object_name']}",
            "value_read_by_pandas": repr(df.at[row_index, "code"]),
            "recovered_code": formula_text,
        }
    )
    df.at[row_index, "code"] = formula_text

formula_recovery_count = len(recovered_cells)
print(f"Fix 0: {formula_recovery_count} rows recovered.")
pd.DataFrame(recovered_cells)

### 2.3 Structure and scope

In [ ]:
column_summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "null_rate_%": (df.isna().mean() * 100).round(1),
        "unique_values": df.nunique(dropna=True),
    }
)
column_summary

In [ ]:
print("Period        :", df["date"].min(), "->", df["date"].max())
print("Games         :", df["game_id"].nunique())
print("Students      :", df["student_id"].nunique())
print("Groups        :", sorted(df["group_id"].dropna().unique()))
print("Levels        :", sorted(df["level_id"].unique()))
print("Duplicated ids:", int(df["id"].duplicated().sum()))

print("\nAction distribution:")
print(df["action_name"].value_counts())

In [ ]:
# Most frequent action / object pairs
(
    df.groupby(["action_name", "object_name"], dropna=False)
    .size()
    .sort_values(ascending=False)
    .head(20)
    .rename("count")
    .to_frame()
)

## 3. Fix 1: mojibake repair

In [ ]:
# C1 control characters produced by the bad CP1252 decoding -> expected replacement
MOJIBAKE_REPLACEMENTS: dict[str, str] = {
    "\u0092": "'",    # mis-decoded curly apostrophe
    "\u0085": "...",  # mis-decoded ellipsis
    "\u009c": "œ",    # mis-decoded oe ligature
}

code_before_fix = df["code"].fillna("")
mojibake_diagnosis = pd.DataFrame(
    [
        {
            "character": f"U+{ord(broken):04X}",
            "replacement": repr(correct),
            "rows_affected": int(code_before_fix.str.contains(broken, regex=False).sum()),
            "occurrences": int(code_before_fix.str.count(broken).sum()),
        }
        for broken, correct in MOJIBAKE_REPLACEMENTS.items()
    ]
)
mojibake_diagnosis

In [ ]:
def fix_mojibake(value):
    # Apply the repair mapping; non-string values are returned unchanged.
    if not isinstance(value, str):
        return value
    for broken, correct in MOJIBAKE_REPLACEMENTS.items():
        value = value.replace(broken, correct)
    return value


repaired_code = df["code"].map(fix_mojibake)
mojibake_mask = repaired_code.fillna("") != df["code"].fillna("")
mojibake_row_count = int(mojibake_mask.sum())

df["code"] = repaired_code
print(f"Fix 1: {mojibake_row_count} rows changed.")

In [ ]:
# Before / after example (excerpt around the first repaired occurrence)
example_index = mojibake_mask.idxmax()
before = code_before_fix[example_index]
after = df.at[example_index, "code"]
position = min(
    (before.find(c) for c in MOJIBAKE_REPLACEMENTS if before.find(c) != -1),
    default=0,
)
start, end = max(0, position - 60), position + 60
print("BEFORE:", repr(before[start:end]))
print("AFTER :", repr(after[start:end]))

In [ ]:
# Check: no residual C1 control character left in `code`
residuals = {
    f"U+{ord(c):04X}": int(df["code"].fillna("").str.count(c).sum())
    for c in MOJIBAKE_REPLACEMENTS
}
assert all(n == 0 for n in residuals.values()), residuals
print("OK, no corrupted character left:", residuals)

## 4. Fix 2: backfilling empty copy/paste captures

The same capture-timing issue affects both sides of a copy/paste, so the rule is applied in both
directions, to every copied panel (not only the code editor):

- an empty `COPIED` trace takes the `code` of the next `PASTED > CONTENT` trace;
- an empty `PASTED > CONTENT` trace takes the `code` of the previous `COPIED` trace.

In both cases the walk stays **inside the same game**, follows chronological order (`date`, then
`id`), and stops as soon as another trace of the same kind is met (another `COPIED` when walking
forward, another `PASTED > CONTENT` when walking backward), so that a different copy is never
attributed to this one. A value made only of whitespace counts as an empty capture.

Both directions read the **pre-fix** values, so a backfilled value is never itself used as a
source: a pair of empty `COPIED` / `PASTED` traces stays empty rather than propagating.

In [ ]:
def is_empty_code(value) -> bool:
    return not isinstance(value, str) or value.strip() == ""


is_copy = df["action_name"] == "COPIED"
is_paste = (df["action_name"] == "PASTED") & (df["object_name"] == "CONTENT")
empty_before_fix = df["code"].map(is_empty_code)

capture_summary = pd.DataFrame(
    [
        {"trace": "COPIED (any panel)", "total": int(is_copy.sum()),
         "empty": int((is_copy & empty_before_fix).sum())},
        {"trace": "PASTED > CONTENT", "total": int(is_paste.sum()),
         "empty": int((is_paste & empty_before_fix).sum())},
    ]
)
capture_summary

In [ ]:
# Look for unambiguous backfills, game by game, reading only pre-fix values
backfills: list[dict] = []

df_ordered = df.sort_values(["date", "id"], kind="mergesort")

for game_id, game in df_ordered.groupby("game_id", sort=False):
    traces = list(
        game[["action_name", "object_name", "code"]].itertuples(index=True, name=None)
    )

    for position, (idx, action, obj, code_value) in enumerate(traces):
        if not is_empty_code(code_value):
            continue

        if action == "COPIED":
            # forward: the next PASTED > CONTENT of the same game
            for other_idx, other_action, other_obj, other_code in traces[position + 1 :]:
                if other_action == "COPIED":
                    break  # another copy happens first: ambiguous attribution
                if other_action == "PASTED" and other_obj == "CONTENT":
                    if not is_empty_code(other_code):
                        backfills.append(
                            {"direction": "COPIED <- PASTED", "game_id": game_id,
                             "object_name": obj, "target_index": idx,
                             "source_index": other_idx, "code": other_code}
                        )
                    break

        elif action == "PASTED" and obj == "CONTENT":
            # backward: the previous COPIED of the same game
            for other_idx, other_action, other_obj, other_code in reversed(traces[:position]):
                if other_action == "PASTED" and other_obj == "CONTENT":
                    break  # another paste happens first: ambiguous attribution
                if other_action == "COPIED":
                    if not is_empty_code(other_code):
                        backfills.append(
                            {"direction": "PASTED <- COPIED", "game_id": game_id,
                             "object_name": other_obj, "target_index": idx,
                             "source_index": other_idx, "code": other_code}
                        )
                    break

backfill_count = len(backfills)
backfill_detail = pd.DataFrame(backfills)
print(f"Fix 2: {backfill_count} rows backfilled.")
(
    backfill_detail.groupby(["direction", "object_name"])
    .size()
    .rename("rows")
    .to_frame()
)

In [ ]:
for backfill in backfills[:3]:
    excerpt = backfill["code"].replace("\n", " ⏎ ")[:100]
    print(f"game {backfill['game_id']} | {backfill['direction']} | "
          f"#{backfill['target_index']} <- #{backfill['source_index']} | {excerpt}")

In [ ]:
backfill_targets = [backfill["target_index"] for backfill in backfills]
assert len(set(backfill_targets)) == len(backfill_targets)  # one source per empty capture

for backfill in backfills:
    df.at[backfill["target_index"], "code"] = backfill["code"]

still_empty_capture = pd.DataFrame(
    [
        {"trace": "COPIED (any panel)", "still_empty": int((is_copy & df["code"].map(is_empty_code)).sum())},
        {"trace": "PASTED > CONTENT", "still_empty": int((is_paste & df["code"].map(is_empty_code)).sum())},
    ]
)
still_empty = int(still_empty_capture["still_empty"].sum())
print(f"Copy/paste traces still empty: {still_empty}")
still_empty_capture

In [ ]:
# Why the remaining captures could not be backfilled
def diagnose_remaining(dataframe: pd.DataFrame) -> pd.DataFrame:
    reasons: list[str] = []

    for _, game in dataframe.sort_values(["date", "id"], kind="mergesort").groupby(
        "game_id", sort=False
    ):
        traces = list(game[["action_name", "object_name", "code"]].itertuples(index=True, name=None))

        for position, (_, action, obj, code_value) in enumerate(traces):
            if not is_empty_code(code_value):
                continue

            if action == "COPIED":
                reason = "no paste afterwards"
                for _, other_action, other_obj, other_code in traces[position + 1 :]:
                    if other_action == "COPIED":
                        reason = "another copy first (ambiguous)"
                        break
                    if other_action == "PASTED" and other_obj == "CONTENT":
                        reason = "next paste also empty"
                        break
                reasons.append(f"COPIED: {reason}")

            elif action == "PASTED" and obj == "CONTENT":
                reason = "no copy before"
                for _, other_action, other_obj, other_code in reversed(traces[:position]):
                    if other_action == "PASTED" and other_obj == "CONTENT":
                        reason = "another paste first (ambiguous)"
                        break
                    if other_action == "COPIED":
                        reason = "previous copy also empty"
                        break
                reasons.append(f"PASTED: {reason}")

    return pd.Series(reasons).value_counts().rename("rows").to_frame()


diagnose_remaining(df)

## 5. Consistency checks

The cleaning must have touched **only** the `code` column, leaving the same rows in the same
order.

In [ ]:
assert df.shape == df_raw.shape, (df.shape, df_raw.shape)
assert list(df.columns) == list(df_raw.columns)
assert df.index.equals(df_raw.index)

changed_columns = [
    column
    for column in df.columns
    if not df[column].fillna("").astype(str).equals(df_raw[column].fillna("").astype(str))
]
assert changed_columns == ["code"], changed_columns

changed_rows = int((df["code"].fillna("") != df_raw["code"].fillna("")).sum())
print("Changed columns:", changed_columns)
print("Changed rows   :", changed_rows,
      f"(formula recovery: {formula_recovery_count}, mojibake: {mojibake_row_count}, "
      f"backfills: {backfill_count})")
print(backfill_detail["direction"].value_counts().to_string())

## 6. LLM service quality and exclusion from the export

The assistant is only available in groups B and C. A request that never got an answer is an
infrastructure failure, not a feedback: there is nothing for an expert to rate, and the replay
shows a request followed by nothing. A game where this happened too often stops informing us about
feedback quality and starts informing us about the reliability of the service.

Requests and answers are paired chronologically within a game and a level, first in first out. A
game is flagged when **more than one third of its requests went unanswered**. The threshold is a
convention, fixed before looking at any expert rating, and applied identically to both groups.

It sits on a sensitive spot: three games land exactly on one third and are therefore kept, and
nothing at all falls between 16.3 and 33.3 per cent. Any threshold inside that band would flag
those three as well.

The traces of a flagged game are left out of the export entirely: they would be replayed, and a
replay that mostly shows requests without answers measures the infrastructure rather than the
feedback. Everything downstream, session traces included, is built from the retained games.

In [ ]:
MAX_LOST_REQUEST_RATE = 1 / 3

assistant = df[df["object_name"] == "ASSISTANT_HELP"].sort_values(["date", "id"], kind="mergesort")

answered: dict[int, bool] = {}
for _, level_traces in assistant.groupby(["game_id", "level_id"], sort=False):
    waiting: list[int] = []
    for trace in level_traces.itertuples():
        if trace.action_name == "ASKED":
            waiting.append(trace.Index)
            answered[trace.Index] = False
        elif trace.action_name == "RECEIVED" and waiting:
            answered[waiting.pop(0)] = True

requests = assistant[assistant["action_name"] == "ASKED"].copy()
requests["answered"] = requests.index.map(answered)

service = (
    requests.groupby("game_id")
    .agg(
        group_id=("group_id", "first"),
        student_id=("student_id", "first"),
        requests=("id", "size"),
        answered=("answered", "sum"),
    )
    .reset_index()
)
service["lost"] = service["requests"] - service["answered"]
service["lost_rate"] = (service["lost"] / service["requests"]).round(3)
service["excluded"] = service["lost_rate"] > MAX_LOST_REQUEST_RATE
service = service.sort_values(["lost_rate", "game_id"], ascending=[False, True])

print(f"{len(service)} games with at least one request, "
      f"{int(service['requests'].sum())} requests, {int(service['lost'].sum())} lost")
service[service["lost"] > 0]

In [ ]:
excluded_games = service.loc[service["excluded"], "game_id"]
kept = service[~service["excluded"]]

print(f"Games flagged (lost rate over {MAX_LOST_REQUEST_RATE:.1%}): {len(excluded_games)}")
print(f"  feedbacks lost with them: {int(service.loc[service['excluded'], 'answered'].sum())} "
      f"out of {int(service['answered'].sum())}")
print("  games per group, before and after:")
print(
    pd.DataFrame(
        {
            "before": df.drop_duplicates("game_id")["group_id"].value_counts(),
            "flagged": service[service["excluded"]]["group_id"].value_counts(),
        }
    )
    .fillna(0).astype(int)
    .assign(after=lambda t: t["before"] - t["flagged"])
    .to_string()
)

service[service["excluded"]]

In [ ]:
retained = df[~df["game_id"].isin(set(excluded_games))].copy()

print(f"{df['game_id'].nunique() - retained['game_id'].nunique()} games removed, "
      f"{len(df) - len(retained):,} traces")
print(f"kept: {retained['game_id'].nunique()} games, {len(retained):,} traces")

## 7. Session traces

The dataset carries no notion of session: a game runs over several class periods, and only the
timestamps say where one period ends. The gap between two consecutive traces of a game is sharply
bimodal, with an empty band in between, so the cut is unambiguous: gaps of at most 45 minutes on
one side, gaps of at least 4 hours on the other, and nothing in between. Any threshold inside that
band produces the same segmentation, and it coincides with the calendar-day rule.

The cut is corroborated by the traces themselves: almost every session already opens with a level
lifecycle trace, `STARTED` for the first one and `RESUMED` afterwards.

Two synthetic traces are added per session, `STARTED > SESSION_n` and `ENDED > SESSION_n`. They are
placed one millisecond before the first trace and one millisecond after the last one, so they frame
the session whatever the tie-breaking rule applied downstream. They carry the level, game time and
progression of the trace they are adjacent to, and the closing trace carries the session length in
seconds in `duration`.

These rows are the only ones in the file that the game client did not produce. They are identified by an
`object_name` of the form `SESSION_1`, `SESSION_2`, `SESSION_3`, numbered in chronological order
within the game, and by an `id` above `SESSION_ID_OFFSET`, which keeps them out of the range of the
source identifiers. The importer must know every one of those object names, otherwise the whole
import fails on an unknown name.

In [ ]:
SESSION_GAP = pd.Timedelta(hours=1)          # anywhere in the empty 45 min to 4 h band
SESSION_BOUNDARY = pd.Timedelta(milliseconds=1)
SESSION_ID_OFFSET = 900_000_000

# Distribution of the gaps between consecutive traces of a game, the basis for the threshold
ordered = retained.sort_values(["date", "id"], kind="mergesort")
gaps = ordered.groupby("game_id")["date"].diff().dt.total_seconds()
gap_bins = [0, 300, 1200, 2700, 14400, 86400, float("inf")]
gap_labels = ["< 5 min", "5 to 20 min", "20 to 45 min", "45 min to 4 h", "4 h to 1 day", "> 1 day"]
(
    pd.cut(gaps.dropna(), bins=gap_bins, labels=gap_labels, right=False)
    .value_counts()
    .reindex(gap_labels)
    .rename("gaps")
    .to_frame()
)

In [ ]:
# One session per uninterrupted stretch of activity, numbered from 1 within each game
session_start = gaps.isna() | (gaps > SESSION_GAP.total_seconds())
ordered = ordered.assign(session_index=session_start.groupby(ordered["game_id"]).cumsum().astype(int))

sessions = (
    ordered.groupby(["game_id", "session_index"], sort=False)
    .agg(
        start=("date", "first"),
        end=("date", "last"),
        first_level=("level_id", "first"),
        last_level=("level_id", "last"),
        first_game_time=("game_time", "first"),
        last_game_time=("game_time", "last"),
        first_progression=("progression", "first"),
        last_progression=("progression", "last"),
        group_id=("group_id", "first"),
        student_id=("student_id", "first"),
        traces=("id", "size"),
    )
    .reset_index()
)
sessions["length_min"] = (sessions["end"] - sessions["start"]).dt.total_seconds() / 60

print(f"{len(sessions)} sessions over {sessions['game_id'].nunique()} games")
print("sessions per game :", sessions.groupby("game_id").size().value_counts().sort_index().to_dict())
print("length in minutes : median %.0f | p90 %.0f | max %.0f"
      % (sessions["length_min"].median(), sessions["length_min"].quantile(0.9), sessions["length_min"].max()))
print("traces per session: median %.0f | min %d | max %d"
      % (sessions["traces"].median(), sessions["traces"].min(), sessions["traces"].max()))
print("sessions spanning two calendar days :",
      int((sessions["start"].dt.date != sessions["end"].dt.date).sum()))

In [ ]:
# Which action already opens each session, as a check on the segmentation
opening_actions = (
    ordered.groupby(["game_id", "session_index"], sort=False)
    .head(1)["action_name"]
    .value_counts()
    .rename("sessions")
    .to_frame()
)
opening_actions

In [ ]:
assert int(retained["id"].max()) < SESSION_ID_OFFSET  # synthetic ids stay out of the source range

session_rows: list[dict] = []
for session in sessions.sort_values(["start"]).itertuples(index=False):
    shared = {
        "game_id": session.game_id,
        "object_name": f"SESSION_{session.session_index}",
        "group_id": session.group_id,
        "student_id": session.student_id,
    }
    session_rows.append(
        {
            **shared,
            "date": session.start - SESSION_BOUNDARY,
            "action_name": "STARTED",
            "level_id": session.first_level,
            "game_time": session.first_game_time,
            "progression": session.first_progression,
        }
    )
    session_rows.append(
        {
            **shared,
            "date": session.end + SESSION_BOUNDARY,
            "action_name": "ENDED",
            "level_id": session.last_level,
            "game_time": session.last_game_time,
            "progression": session.last_progression,
            "duration": round((session.end - session.start).total_seconds()),
        }
    )

session_traces = pd.DataFrame(session_rows).reindex(columns=retained.columns)
session_traces = session_traces.sort_values("date", kind="mergesort").reset_index(drop=True)
session_traces["id"] = SESSION_ID_OFFSET + session_traces.index + 1

print(f"{len(session_traces)} session traces added for {len(sessions)} sessions")
print(session_traces.groupby(["object_name", "action_name"]).size().rename("traces").to_string())
session_traces.head(4)

## 8. Formatting for the webtool importer

The importer parses the CSV with `csv-parse` and expects exactly the columns of
`packages/db/data/csv/cleaned_traces.csv`, in this order, with these conventions:

| Column(s) | Format |
|---|---|
| `id` | integer, unique in the file (dedup/link key at import) |
| `date` | `YYYY-MM-DD HH:MM:SS.SSS`, read as French summer time (UTC+2), no explicit offset |
| `game_id`, `group_id`, `student_id` | required, non-empty strings |
| `level_id`, `game_time`, `progression` | required integers |
| `duration`, `value`, `x_pos`, `y_pos` | optional integers (empty string = absent) |
| `code` | optional; real line breaks written as the literal two-character sequence `\n` |
| `reason_name`, `error_message` | optional strings (empty = absent) |
| `flipped`, `owned_key` | optional booleans, `TRUE` / `FALSE` only (empty = absent) |

`group_id` also drives the feedback modality on the importer side: `B` → FREE, `C` →
CONSTRAINED, anything else → CONTROL.

The source records every timestamp to the millisecond and the export keeps that precision, which
the ordering depends on: within a single game, 23 603 traces share their timestamp with another
trace when the value is truncated to the second, against 8 355 when the milliseconds are kept. The
importer must accept the `.SSS` suffix and store the column with millisecond precision.

Rows are written in chronological order (`date`, then `id`), which is the order the timeline
displays.

In [ ]:
EXPORT_COLUMNS = [
    "id", "date", "game_id", "level_id", "game_time", "progression", "action_name",
    "object_name", "duration", "code", "value", "reason_name", "error_message",
    "x_pos", "y_pos", "flipped", "owned_key", "group_id", "student_id",
]
REQUIRED_INTEGER_COLUMNS = ["id", "level_id", "game_time", "progression"]
OPTIONAL_INTEGER_COLUMNS = ["duration", "value", "x_pos", "y_pos"]
BOOLEAN_COLUMNS = ["flipped", "owned_key"]
TEXT_COLUMNS = ["game_id", "group_id", "student_id", "action_name", "object_name",
                "reason_name", "error_message"]
DATE_FORMAT = "%Y-%m-%d %H:%M:%S.%f"


def format_date(values: pd.Series) -> pd.Series:
    # The source is millisecond-precise, so cutting the last three digits of %f loses nothing.
    assert (values.dt.microsecond % 1000 == 0).all()
    return values.dt.strftime(DATE_FORMAT).str[:-3]


def format_integer(value) -> str:
    return "" if pd.isna(value) else str(int(value))


def format_boolean(value) -> str:
    return "" if pd.isna(value) else ("TRUE" if bool(value) else "FALSE")


def format_text(value) -> str:
    return "" if pd.isna(value) else str(value)


def format_code(value) -> str:
    # The importer turns the literal two-character sequence \n back into a line break.
    if not isinstance(value, str):
        return ""
    return value.replace("\r\n", "\n").replace("\r", "\n").replace("\n", "\\n")


to_export = (
    pd.concat([retained, session_traces], ignore_index=True)
    .sort_values(["date", "id"], kind="mergesort")
    .reset_index(drop=True)
)

export_df = pd.DataFrame(index=to_export.index)
export_df["date"] = format_date(to_export["date"])
export_df["code"] = to_export["code"].map(format_code)
for column in REQUIRED_INTEGER_COLUMNS + OPTIONAL_INTEGER_COLUMNS:
    export_df[column] = to_export[column].map(format_integer)
for column in BOOLEAN_COLUMNS:
    export_df[column] = to_export[column].map(format_boolean)
for column in TEXT_COLUMNS:
    export_df[column] = to_export[column].map(format_text)

export_df = export_df[EXPORT_COLUMNS]
print(f"{len(export_df):,} rows to export "
      f"({len(retained):,} game traces + {len(session_traces):,} session traces)")
export_df.head(3)

## 9. Format validation

Every rule the importer enforces is checked here, so that a rejected import cannot be a surprise.

In [ ]:
INTEGER_PATTERN = re.compile(r"^-?\d+$")
DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3}$")

assert list(export_df.columns) == EXPORT_COLUMNS, list(export_df.columns)
# Every column went through a formatting function, so nothing numeric or dated is left. The dtype
# itself depends on the pandas version (object, or str once string inference is on), hence the
# check on the kind rather than on the exact dtype.
for column in EXPORT_COLUMNS:
    assert pd.api.types.is_string_dtype(export_df[column]), (column, export_df[column].dtype)

# Unique, well-formed ids
assert not export_df["id"].duplicated().any()

# Required values: never empty
for column in ["id", "date", "game_id", "level_id", "game_time", "progression",
               "action_name", "object_name", "group_id", "student_id"]:
    empty_count = int((export_df[column].str.strip() == "").sum())
    assert empty_count == 0, (column, empty_count)

# Types
assert export_df["date"].str.match(DATE_PATTERN).all()
for column in REQUIRED_INTEGER_COLUMNS:
    assert export_df[column].str.match(INTEGER_PATTERN).all(), column
for column in OPTIONAL_INTEGER_COLUMNS:
    values = export_df[column]
    assert (values.str.match(INTEGER_PATTERN) | (values == "")).all(), column
for column in BOOLEAN_COLUMNS:
    unexpected = set(export_df[column].unique()) - {"TRUE", "FALSE", ""}
    assert not unexpected, (column, unexpected)

# No raw line break left in any field (they must be the literal \n sequence)
assert not export_df["code"].str.contains("\n|\r", regex=True).any()

# One game_id belongs to exactly one (group_id, student_id) pair
per_game = export_df.groupby("game_id")[["group_id", "student_id"]].nunique()
assert (per_game == 1).all().all(), per_game[(per_game != 1).any(axis=1)]

# Rows are in the order the timeline displays
assert export_df["date"].is_monotonic_increasing

print("OK, the export matches the importer format.")

In [ ]:
# Session traces: one opening and one closing trace per session, properly nested
is_session = export_df["object_name"].str.startswith("SESSION_")
session_export = export_df[is_session]
assert len(session_export) == 2 * len(sessions)
assert (session_export["id"].astype(int) > SESSION_ID_OFFSET).all()

for game_id, game in session_export.groupby("game_id"):
    actions = list(game["action_name"])
    assert actions == ["STARTED", "ENDED"] * (len(actions) // 2), (game_id, actions)
    # sessions are numbered from 1, in chronological order, each number used exactly twice
    numbers = [int(name.removeprefix("SESSION_")) for name in game["object_name"]]
    assert numbers == sorted(numbers), (game_id, numbers)
    assert set(numbers) == set(range(1, len(numbers) // 2 + 1)), (game_id, numbers)

# Every game trace sits inside one of its sessions
game_export = export_df[~is_session]
bounds = session_export.pivot_table(
    index=["game_id", session_export.groupby("game_id").cumcount() // 2],
    columns="action_name", values="date", aggfunc="first",
)
assert bounds["STARTED"].lt(bounds["ENDED"]).all()
assert game_export["date"].min() > session_export["date"].min()
assert game_export["date"].max() < session_export["date"].max()

print(f"OK, {len(sessions)} sessions framed by {len(session_export)} traces.")
session_export.head(4)

In [ ]:
# How many traces still share a timestamp with another trace of the same game
timestamp_collisions = pd.DataFrame(
    [
        {"precision": "second",
         "traces sharing a timestamp": int(
             export_df.assign(stamp=export_df["date"].str.slice(0, 19))
             .duplicated(["game_id", "stamp"], keep=False).sum())},
        {"precision": "millisecond",
         "traces sharing a timestamp": int(
             export_df.duplicated(["game_id", "date"], keep=False).sum())},
    ]
)
print(timestamp_collisions.to_string(index=False), "\n")

MODALITY_BY_GROUP = {"B": "FREE", "C": "CONSTRAINED"}

games = export_df.drop_duplicates("game_id")
modality_summary = (
    games.assign(modality=games["group_id"].map(lambda g: MODALITY_BY_GROUP.get(g, "CONTROL")))
    .groupby(["group_id", "modality"])
    .size()
    .rename("games")
    .reset_index()
)
print(modality_summary.to_string(index=False), "\n")

assistant_help = export_df[export_df["object_name"] == "ASSISTANT_HELP"]
print("ASSISTANT_HELP traces (paired into LlmFeedback at import):")
print(assistant_help["action_name"].value_counts().to_string())

## 10. Export

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
export_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8", lineterminator="\n")
size_mb = OUTPUT_PATH.stat().st_size / 1024**2
print(f"Written: {OUTPUT_PATH.relative_to(PROJECT_ROOT)} ({size_mb:.1f} MB, {len(export_df):,} rows)")

# Read back as a sanity check, and confirm the round-trip of the escaped line breaks
reloaded = pd.read_csv(OUTPUT_PATH, dtype=str, keep_default_na=False, low_memory=False)
assert list(reloaded.columns) == EXPORT_COLUMNS
assert len(reloaded) == len(export_df)
# astype("object") on both sides so that the comparison does not depend on the string dtype in use
assert reloaded.astype("object").equals(export_df.reset_index(drop=True).astype("object"))
assert (
    reloaded["code"].str.replace("\\n", "\n", regex=False)
    == to_export["code"].fillna("").reset_index(drop=True)
).all()
print("Read-back OK:", reloaded.shape)

## 11. Summary

In [ ]:
summary = pd.DataFrame(
    [
        {"step": "Input rows", "value": len(df_raw)},
        {"step": "Fix 0: Excel formula cells recovered", "value": formula_recovery_count},
        {"step": "Fix 1: mojibake (rows)", "value": mojibake_row_count},
        {"step": "Fix 2: copy/paste backfills (rows)", "value": backfill_count},
        {"step": "Fix 2: of which COPIED <- PASTED",
         "value": int((backfill_detail["direction"] == "COPIED <- PASTED").sum())},
        {"step": "Fix 2: of which PASTED <- COPIED",
         "value": int((backfill_detail["direction"] == "PASTED <- COPIED").sum())},
        {"step": "Copy/paste captures still empty", "value": still_empty},
        {"step": "Rows changed in total", "value": changed_rows},
        {"step": "Sessions", "value": len(sessions)},
        {"step": "Session traces added", "value": len(session_traces)},
        {"step": "Output rows", "value": len(export_df)},
        {"step": "Games flagged on LLM service quality", "value": int(service["excluded"].sum())},
    ]
)
summary